# 26. t-SNE
**Use case:** Financial Services — Visualising card-transaction behaviour to reveal fraud clusters

A card issuer's fraud investigators review thousands of alerts a week and suspect that some fraud comes in organised rings that look alike. Each card has dozens of behavioural features (spend amount, night-time share, foreign share, velocity, merchant diversity), which nobody can picture. A t-SNE map gives investigators a 2-D picture where similar cards sit together, so a tight island of confirmed fraud can be reviewed in bulk and new rules drafted. Success means fraud types form clean islands (high neighbourhood purity) and investigators find rings faster, without anyone treating map distances as a score.

> 💡 Imagine seating 500 wedding guests at round tables so that people who know each other sit together. You do not care how far apart table 3 and table 17 are; you only care that friends end up at the same table. t-SNE is that seating planner for data points.

Legend in code: `# 🐍 PY:` Python lesson · `# 🧠 ML:` ML lesson

In [ ]:
# Run once (Colab already has most of these)
!pip -q install numpy pandas scikit-learn scipy xgboost torch

## CONCEPT: Neighbour probabilities for 6 cards by hand
Compute t-SNE's similarity tables for 6 cards in plain numpy and watch a few gradient steps pull fraud-like cards together.

| 5W + H | |
|---|---|
| **Who** | A new fraud analyst who wants to understand what t-SNE actually calculates. |
| **What** | Six cards with two features (average spend, night-time share): three normal, three fraud-like. |
| **When** | Before trusting any t-SNE picture in an investigation pack. |
| **Where** | A plain Python script with numpy only, no machine-learning library. |
| **Why** | Knowing that t-SNE preserves neighbours, not distances, prevents the most common misreading of its maps. |
| **How** | Compute pairwise distances, Gaussian probabilities P, Student-t probabilities Q, KL divergence, then 200 gradient steps. |

**🧠 ML concepts:** Pairwise distance matrix, Neighbour probabilities (Gaussian kernel), Student-t low-dimensional similarity, KL divergence as a loss, Gradient descent on point positions  
**🐍 Python concepts:** Lists of tuples, numpy broadcasting with None axes, In-place functions like np.fill_diagonal, keepdims in reductions, zip and generator expressions in str.join, f-strings with format specs

In [ ]:
# t-SNE by hand: how similar are 6 card customers? (Concept)
import numpy as np

np.set_printoptions(precision=3, suppress=True)
# 🐍 PY: a list of tuples -> each tuple is one customer: (avg txn $ in hundreds, night-time txn share)
customers = [(0.5, 0.10), (0.6, 0.12), (0.55, 0.08), (4.0, 0.80), (4.2, 0.85), (3.9, 0.75)]
labels = ["N1", "N2", "N3", "F1", "F2", "F3"]  # N = normal, F = fraud-like
# 🐍 PY: np.array turns the list into a 6x2 matrix we can do maths on
X = np.array(customers)

# 🧠 ML: step 1 - squared distance between every pair of customers
# 🐍 PY: broadcasting X[:, None, :] - X[None, :, :] makes a 6x6x2 grid of differences
D = ((X[:, None, :] - X[None, :, :]) ** 2).sum(axis=2)
print("Squared distances:\n", D)

# 🧠 ML: step 2 - turn distance into 'probability I would pick you as my neighbour' (Gaussian)
sigma = 1.0  # real t-SNE tunes sigma per point using perplexity; we fix it to keep it simple
P_cond = np.exp(-D / (2 * sigma ** 2))
# 🐍 PY: np.fill_diagonal edits the matrix in place - a point is never its own neighbour
np.fill_diagonal(P_cond, 0.0)
# 🐍 PY: keepdims=True keeps shape (6,1) so each row divides by its own sum
P_cond = P_cond / P_cond.sum(axis=1, keepdims=True)
# 🧠 ML: step 3 - symmetrise so p_ij = p_ji, then normalise the whole matrix to sum to 1
P = (P_cond + P_cond.T) / (2 * len(X))
print("\nHigh-dim similarity P (rows sum ~1/6):\n", P)

# 🧠 ML: step 4 - a guess for the 2-D map; t-SNE uses a heavy-tailed Student-t here
rng = np.random.default_rng(42)
Y = rng.normal(0, 1, size=(6, 2))
DY = ((Y[:, None, :] - Y[None, :, :]) ** 2).sum(axis=2)
Q = 1.0 / (1.0 + DY)
np.fill_diagonal(Q, 0.0)
Q = Q / Q.sum()
# 🧠 ML: step 5 - KL divergence = how badly the 2-D map betrays the true neighbourhoods
mask = P > 0
kl = np.sum(P[mask] * np.log(P[mask] / Q[mask]))
print(f"\nKL(P||Q) for a random 2-D layout: {kl:.3f}")

# 🧠 ML: step 6 - a few gradient steps pull similar points together, push others apart
for step in range(1, 201):
    DY = ((Y[:, None, :] - Y[None, :, :]) ** 2).sum(axis=2)
    W = 1.0 / (1.0 + DY)
    np.fill_diagonal(W, 0.0)
    Q = W / W.sum()
    # 🧠 ML: the classic t-SNE gradient: 4 * sum_j (p_ij - q_ij) * w_ij * (y_i - y_j)
    grad = 4 * (((P - Q) * W)[:, :, None] * (Y[:, None, :] - Y[None, :, :])).sum(axis=1)
    Y -= 2.0 * grad
    if step in (1, 50, 200):
        kl = np.sum(P[mask] * np.log(P[mask] / Q[mask]))
        print(f"step {step:3d}  KL={kl:.3f}")
print("Final 2-D map:", ", ".join(f"{n}=({y[0]:.2f},{y[1]:.2f})" for n, y in zip(labels, Y)))
# 🐍 PY: zip pairs each label with its row; argmax finds the closest neighbour
for name, row in zip(labels, P):
    best = labels[int(np.argmax(row))]
    print(f"{name}: strongest neighbour = {best}  (p={row.max():.3f})")

**Try this next:**
- Change sigma to 0.1 and to 5.0: how do the P values change, and does the map still separate?
- Move F1 to (1.0, 0.2) so it looks almost normal and rerun: where does it land on the 2-D map?
- Replace the Student-t Q with a Gaussian (np.exp(-DY)) and compare the final KL and spread.

## BUILD: Map 1,200 card customers with scikit-learn t-SNE
Run scikit-learn's TSNE on realistic synthetic card behaviour, draw an ASCII map and measure how cleanly the fraud ring forms its own island.

| 5W + H | |
|---|---|
| **Who** | A fraud analytics team preparing a monthly investigator briefing. |
| **What** | 1,200 cards across four behaviour segments, including a 50-card fraud ring, described by five behavioural features. |
| **When** | After the monthly feature refresh, before the investigator review meeting. |
| **Where** | A Python notebook or batch job using pandas and scikit-learn. |
| **Why** | Investigators can lasso a visible island of similar cards and review them together instead of one alert at a time. |
| **How** | Scale features, run TSNE(perplexity=30), print a character scatter plot, then compute 10-nearest-neighbour purity per segment. |

**🧠 ML concepts:** Feature scaling before distance-based methods, Perplexity as neighbourhood size, KL divergence of the final fit, Using labels only for evaluation, not fitting, k-nearest-neighbour purity as a quantitative check  
**🐍 Python concepts:** Helper functions returning DataFrames, pd.concat with ignore_index, List comprehensions for column lists, List-of-lists as a text canvas, numpy fancy indexing with a 2-D index array, groupby + mean + to_string

In [ ]:
# t-SNE on 1,200 synthetic card customers: can investigators SEE the fraud ring? (Build)
import warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")
rng = np.random.default_rng(42)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 10)

# 🐍 PY: a helper function that builds one customer segment as a DataFrame
def make_segment(n, amt, night, foreign, velocity, merchants, label):
    return pd.DataFrame({
        "avg_txn_amount": rng.normal(amt, amt * 0.4, n),
        "night_txn_share": np.clip(rng.normal(night, 0.12, n), 0, 1),
        "foreign_txn_share": np.clip(rng.normal(foreign, 0.12, n), 0, 1),
        "txn_per_day": rng.normal(velocity, velocity * 0.35, n),
        "distinct_merchants_7d": rng.poisson(merchants, n),
        "segment": label,
    })

# 🧠 ML: four behaviour groups - t-SNE never sees 'segment', we only use it to colour the map afterwards
# 🐍 PY: pd.concat stacks DataFrames vertically; ignore_index renumbers rows 0..n-1
df = pd.concat([
    make_segment(600, 60, 0.05, 0.02, 2.0, 8, "Retail"),
    make_segment(350, 250, 0.10, 0.30, 1.0, 5, "Traveler"),
    make_segment(200, 25, 0.40, 0.01, 6.0, 20, "Student"),
    make_segment(50, 320, 0.30, 0.35, 4.0, 14, "FraudRing"),
], ignore_index=True)
features = [c for c in df.columns if c != "segment"]
print(df.groupby("segment")[features].mean().round(2).to_string())

# 🧠 ML: scale first - otherwise dollars dominate shares and the map just sorts by amount
X = StandardScaler().fit_transform(df[features])

# 🧠 ML: perplexity ~ how many neighbours each point 'cares about' (5-50 typical)
tsne = TSNE(n_components=2, perplexity=30, init="pca", random_state=42)
emb = tsne.fit_transform(X)
print(f"\nFinal KL divergence: {tsne.kl_divergence_:.3f}")

# 🐍 PY: ASCII scatter plot - a list of lists of characters acts as our canvas
def ascii_scatter(emb, labels, width=60, height=18):
    grid = [[" "] * width for _ in range(height)]
    # 🐍 PY: min/max per column, then rescale coordinates to integer grid cells
    lo, hi = emb.min(axis=0), emb.max(axis=0)
    cells = ((emb - lo) / (hi - lo) * [width - 1, height - 1]).astype(int)
    # 🐍 PY: draw the rare class last so it is not hidden under the big ones
    order = np.argsort(labels == "FraudRing")
    for i in order:
        x, y = cells[i]
        grid[height - 1 - y][x] = labels[i][0]
    print("+" + "-" * width + "+")
    for row in grid:
        print("|" + "".join(row) + "|")
    print("+" + "-" * width + "+")

print("\nt-SNE map (R=Retail T=Traveler S=Student F=FraudRing)")
ascii_scatter(emb, df["segment"].to_numpy(dtype=str))

# 🧠 ML: quantitative check - share of each point's 10 nearest map-neighbours with the same segment
nn = NearestNeighbors(n_neighbors=11).fit(emb)
_, idx = nn.kneighbors(emb)
# 🐍 PY: to_numpy(dtype=str) gives a plain numpy string array (fast fancy indexing)
lab = df["segment"].to_numpy(dtype=str)
# 🐍 PY: idx[:, 1:] skips column 0, which is the point itself
purity = (lab[idx[:, 1:]] == lab[:, None]).mean(axis=1)
df["knn_purity"] = purity
print("\n10-NN purity in the 2-D map, by segment:")
print(df.groupby("segment")["knn_purity"].mean().round(3).to_string())

fr = df.loc[df.segment == "FraudRing", "knn_purity"].mean()
print(f"\nBusiness read: {fr:.0%} of a fraud-ring card's map neighbours are also fraud-ring cards,")
print("so investigators can lasso that island and review those accounts together.")

**Try this next:**
- Set perplexity to 5 and then 100: how does the map change and what happens to FraudRing purity?
- Remove the StandardScaler line (use df[features].values) and see which segments merge.
- Shrink the fraud ring to 15 cards: is it still visible as an island?

## PRODUCTION: Investigator-grade fraud map: PCA, perplexity sweep, trust checks
Build a reproducible pipeline that pre-reduces with PCA, sweeps perplexity, scores maps with trustworthiness and kNN purity, compares with PCA 2-D and saves the artefacts.

| 5W + H | |
|---|---|
| **Who** | A fraud data-science team delivering maps that investigators and model-risk reviewers will scrutinise. |
| **What** | 805 cards, 20 behavioural features (6 informative, 14 noise), five segments including account-takeover and card-testing fraud. |
| **When** | Monthly, or whenever a new fraud pattern is suspected and the team wants a visual hypothesis. |
| **Where** | A scheduled batch job; the embedding and preprocessing pipeline are versioned to a model store. |
| **Why** | A single t-SNE run can mislead. Sweeping settings and scoring the map shows whether the islands are real or an artefact. |
| **How** | Pipeline(StandardScaler, PCA 10) then TSNE at perplexity 5/30/50, choose by trustworthiness + purity, compare with PCA 2-D, audit fraud cards outside islands, save with joblib. |

**🧠 ML concepts:** PCA pre-reduction before t-SNE, Perplexity sensitivity analysis, Trustworthiness as map-quality metric, kNN purity per class, Linear (PCA) vs non-linear (t-SNE) embeddings, Why t-SNE has no transform for new data  
**🐍 Python concepts:** Module-level constants and dicts of tuples, Functions with docstrings and default arguments, sklearn Pipeline and named_steps, Building a results table from a list of dicts, np.isin boolean masks, if __name__ == '__main__' entry point

In [ ]:
# t-SNE for fraud investigators: PCA pre-reduction, perplexity sweep, trust checks, PCA comparison (Production)
import time
import tempfile
import warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, trustworthiness
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")
SEED = 42
rng = np.random.default_rng(SEED)
SEGMENTS = {  # name: (count, map letter)
    "Retail": (400, "R"), "Traveler": (200, "T"), "Student": (140, "S"),
    "AccountTakeover": (40, "A"), "CardTesting": (25, "C"),
}
N_FEATURES = 20


def make_data():
    """20 behavioural features; only 6 of them actually separate the segments (rest is noise)."""
    # 🧠 ML: realistic data is high-dimensional and noisy - that is exactly why we pre-reduce with PCA
    # 🐍 PY: one row of 6 'signal' means per segment (amount, night, foreign, velocity, mcc, decline)
    signal = np.array([[0, 0, 0, 0, 0, 0], [2.5, 0, 3, -1, 0, 0], [-1.5, 2.5, 0, 2, 1, 0],
                       [3, 2, 2.5, 1.5, 2.5, 1], [-2, 1.5, 0, 4, 3, 3.5]], dtype=float)
    frames = []
    for k, (name, (n, _)) in enumerate(SEGMENTS.items()):
        X = rng.normal(0, 1, size=(n, N_FEATURES))
        # 🐍 PY: slice assignment adds the segment's signal to the first 6 columns only
        X[:, :6] += signal[k]
        f = pd.DataFrame(X, columns=[f"feat_{i:02d}" for i in range(N_FEATURES)])
        f["segment"] = name
        frames.append(f)
    df = pd.concat(frames, ignore_index=True)
    names = ["txn_amount_z", "night_share_z", "foreign_share_z", "velocity_z", "mcc_entropy_z", "decline_rate_z"]
    # 🐍 PY: dict(zip(...)) builds a rename map old->new in one line
    return df.rename(columns=dict(zip(df.columns[:6], names)))


def knn_purity(emb, labels, k=10):
    """Share of each point's k nearest map-neighbours that share its label."""
    # 🧠 ML: purity measures how well known classes form islands - it is a sanity check, not a model metric
    _, idx = NearestNeighbors(n_neighbors=k + 1).fit(emb).kneighbors(emb)
    return (labels[idx[:, 1:]] == labels[:, None]).mean(axis=1)


def ascii_scatter(emb, labels, letters, width=64, height=18):
    grid = [[" "] * width for _ in range(height)]
    lo, hi = emb.min(axis=0), emb.max(axis=0)
    cells = ((emb - lo) / (hi - lo + 1e-9) * [width - 1, height - 1]).astype(int)
    # 🐍 PY: sort so the big 'Retail' class is drawn first and rare fraud classes stay visible on top
    order = np.argsort([labels[i] != "Retail" for i in range(len(labels))], kind="stable")
    for i in order:
        x, y = cells[i]
        grid[height - 1 - y][x] = letters[labels[i]]
    print("+" + "-" * width + "+")
    for row in grid:
        print("|" + "".join(row) + "|")
    print("+" + "-" * width + "+")


def main():
    df = make_data()
    feats = [c for c in df.columns if c != "segment"]
    labels = df["segment"].to_numpy(dtype=str)
    letters = {k: v[1] for k, v in SEGMENTS.items()}
    # 🐍 PY: value_counts().to_dict() turns the class counts into a compact dict for printing
    print(f"Customers: {len(df)}, features: {len(feats)}, mix: {df['segment'].value_counts().to_dict()}")

    # 🧠 ML: Pipeline = scale -> PCA to 10 dims. PCA strips noise and makes t-SNE faster and more stable
    prep = Pipeline([("scale", StandardScaler()), ("pca", PCA(n_components=10, random_state=SEED))])
    Z = prep.fit_transform(df[feats])
    ev = prep.named_steps["pca"].explained_variance_ratio_
    print(f"\nPCA(10) keeps {ev.sum():.1%} of variance; first 3 PCs: {np.round(ev[:3], 3)}")

    # 🧠 ML: baseline = plain PCA to 2-D (linear, fast, distances ARE meaningful)
    pca2 = Z[:, :2]
    rows = [{"method": "PCA-2D", "perplexity": "-", "trust@10": trustworthiness(Z, pca2, n_neighbors=10),
             "purity@10": knn_purity(pca2, labels).mean(), "secs": 0.0}]

    # 🧠 ML: perplexity sweep - small = very local view, large = more global structure; no single right value
    best, best_score = None, -1
    for perp in [5, 30, 50]:
        t0 = time.time()
        emb = TSNE(n_components=2, perplexity=perp, init="pca", max_iter=600, random_state=SEED).fit_transform(Z)
        # 🧠 ML: trustworthiness (0-1) penalises map-neighbours that were NOT neighbours in the original space
        tr = trustworthiness(Z, emb, n_neighbors=10)
        pu = knn_purity(emb, labels).mean()
        rows.append({"method": "t-SNE", "perplexity": perp, "trust@10": tr, "purity@10": pu,
                     "secs": time.time() - t0})
        if tr + pu > best_score:
            best, best_score, best_perp = emb, tr + pu, perp
    res = pd.DataFrame(rows)
    print("\nMethod comparison (higher trust/purity = better local fidelity):")
    print(res.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

    print(f"\nBest t-SNE map (perplexity={best_perp}) R=Retail T=Traveler S=Student A=AcctTakeover C=CardTest")
    ascii_scatter(best, labels, letters, height=11)
    print("PCA 2-D map for comparison:")
    ascii_scatter(pca2, labels, letters, height=6)

    # 🧠 ML: per-class purity tells investigators which fraud types form clean islands worth bulk review
    df["purity"] = knn_purity(best, labels)
    # 🐍 PY: groupby -> mean -> round -> to_dict gives {segment: purity} on one printable line
    print("\nPer-segment 10-NN purity on best t-SNE map:", df.groupby("segment")["purity"].mean().round(2).to_dict())

    # 🧠 ML: business triage - fraud cards that do NOT sit in their own island are what a map alone misses
    # 🐍 PY: np.isin tests membership against a list, giving a boolean mask
    is_fraud = np.isin(labels, ["AccountTakeover", "CardTesting"])
    hidden = (is_fraud & (df["purity"].to_numpy() < 0.5)).sum()
    print(f"\nFraud cards whose map neighbours are mostly other segments: {hidden} of {is_fraud.sum()}")

    # 🐍 PY: tempfile.mkdtemp gives a safe scratch folder; joblib stores numpy-heavy objects efficiently
    out = tempfile.mkdtemp()
    joblib.dump({"prep": prep, "embedding": best, "perplexity": best_perp}, f"{out}/tsne_fraud_map.joblib")
    print(f"Saved preprocessing pipeline + embedding to {out}")

    print("\nMODEL-RISK NOTE (SR 11-7 / investigator guidance):")
    print(" - t-SNE is an EXPLORATION tool. Cluster sizes and gaps between islands are NOT meaningful distances.")
    print(" - It has no transform() for new cards: never score live transactions with a t-SNE coordinate.")
    print(" - Use islands to form hypotheses, then confirm with a validated, documented fraud model + case review.")
    print(" - Record perplexity, seed and PCA settings so the picture shown to investigators is reproducible.")


if __name__ == "__main__":
    main()

**Try this next:**
- Change PCA n_components to 3 and to 20: how do trustworthiness and fraud purity move?
- Add perplexity 100 to the sweep and see whether the global layout changes.
- Run with three different random_state values and check whether the same islands appear each time.